# Stipple G3 smoke test — 10M points

10 Gaussian clusters arranged in a ring, 1M points each. Total: 10M points.

Gate target: first render < 5s, FPS > 15. Memory: ~80 MB positions, ~40 MB color codes.

In [1]:
import numpy as np
from stipple import Stipple
import stipple
print(f"stipple {stipple.__version__}")

stipple 0.1.0.dev0


In [2]:
rng = np.random.default_rng(7)
n = 10_000_000
k = 10
n_per = n // k

angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
centers = np.stack([5 * np.cos(angles), 5 * np.sin(angles)], axis=1).astype(np.float32)
labels = np.repeat([f'c{i}' for i in range(k)], n_per)

x = np.empty(n, dtype=np.float32)
y = np.empty(n, dtype=np.float32)
for ki in range(k):
    sl = slice(ki * n_per, (ki + 1) * n_per)
    x[sl] = centers[ki, 0] + rng.standard_normal(n_per).astype(np.float32) * 0.7
    y[sl] = centers[ki, 1] + rng.standard_normal(n_per).astype(np.float32) * 0.7

print(f'{n:,} points · {k} clusters · raw MB: {(x.nbytes + y.nbytes) / 1024 / 1024:.1f}')

10,000,000 points · 10 clusters · raw MB: 76.3


In [3]:
w = Stipple(x=x, y=y, color=labels)
w

In [4]:
import time
for _ in range(200):
    if w.last_fps:
        break
    time.sleep(0.1)
print(f'status         : {w.status}')
print(f'rows_received  : {w.rows_received:,}')
print(f'bytes_received : {w.bytes_received:,} ({w.bytes_received / 1024 / 1024:.1f} MB)')
print(f'categories     : {w.color_categories}')
print(f'FPS            : {w.last_fps:.1f}')
print(f'frame ms       : {w.avg_frame_ms:.2f}')

status         : data-rendered
rows_received  : 10,000,000
bytes_received : 120,000,464 (114.4 MB)
categories     : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
FPS            : 60.1
frame ms       : 16.65
